In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

d:\a1file_install\anaconda_ins\envs\th11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os

In [3]:
os.getcwd()


'e:\\jiebang\\a_proj'

In [4]:
# 1. 配置与模型加载
MODEL_NAME = './qwen_model/Qwen3-Embedding-0.6B'
print(f"正在加载模型: {MODEL_NAME} ...")
model = SentenceTransformer(MODEL_NAME,device='cuda')

正在加载模型: ./qwen_model/Qwen3-Embedding-0.6B ...


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 8689.55it/s]


In [5]:
# 2. 读取数据
df = pd.read_csv("Full_100.csv")
X_raw = df['text'].tolist()
y = df['label'].values

In [ ]:
# 3. 提取向量特征 (Embedding)
# 为 Qwen-instruct 提供指令前缀，能显著提升分类效果
def get_qwen_embeddings(texts):
    # 针对分类任务的指令
    instruction = "判断相关文本是否属于固定资产类型，以便进行准确的文本分类："
    # 拼接指令
    full_texts = [f"{instruction}{t}" for t in texts]
    return model.encode(full_texts, batch_size=256, show_progress_bar=True)
start_time = time.time()  
print("正在提取 100 条数据的向量特征...")
end_time = time.time()          
X_embeddings = get_qwen_embeddings(X_raw)
print(f"编码完成，耗时: {end_time - start_time:.2f} 秒")

正在提取 100 条数据的向量特征...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.37it/s]


In [5]:
# 4. 划分数据集 (70% 训练, 30% 验证)
X_train, X_test, y_train, y_test = train_test_split(
    X_embeddings, y, test_size=0.3, random_state=42, stratify=y
)

In [ ]:
# 5. 训练分类器 KNN-1
# 使用逻辑回归，并开启 class_weight='balanced' 以处理可能的类别轻微不平衡
clf = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000)
clf.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [7]:
# 6. 验证集评估
y_pred = clf.predict(X_test)
print("\n--- 验证集评估报告 (30条数据) ---")
print(classification_report(y_test, y_pred, target_names=['其他类型 (0)', '固定资产 (1)']))
print(f"总体准确率: {accuracy_score(y_test, y_pred):.2%}")


--- 验证集评估报告 (30条数据) ---
              precision    recall  f1-score   support

    其他类型 (0)       0.72      0.93      0.81        14
    固定资产 (1)       0.92      0.69      0.79        16

    accuracy                           0.80        30
   macro avg       0.82      0.81      0.80        30
weighted avg       0.83      0.80      0.80        30

总体准确率: 80.00%


In [8]:
# 7. 应用于海量未标注数据 (假设你有一个列表 unlabeled_list)
def predict_new_data(new_texts):
    print(f"正在处理 {len(new_texts)} 条新数据...")
    new_embeddings = get_qwen_embeddings(new_texts)
    preds = clf.predict(new_embeddings)
    # 获取置信度
    probs = clf.predict_proba(new_embeddings).max(axis=1)
    return preds, probs

In [9]:
# 示例预测
sample_unlabeled = [
    "某某大学办公家具采购合同。办公桌椅。木质家具", 
    "某某局网络安全等级保护测评服务定点采购合同"
]
labels, confidences = predict_new_data(sample_unlabeled)

for txt, lab, conf in zip(sample_unlabeled, labels, confidences):
    print(f"预测结果: {'实物' if lab==1 else '服务'} | 置信度: {conf:.4f} | 文本: {txt[:30]}...")

正在处理 2 条新数据...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.93it/s]

预测结果: 实物 | 置信度: 0.6079 | 文本: 某某大学办公家具采购合同。办公桌椅。木质家具...
预测结果: 服务 | 置信度: 0.5475 | 文本: 某某局网络安全等级保护测评服务定点采购合同...
